# Notebook 02 · Ver la serie

De pandas al autocorrelograma · Series Temporales · GRDAN 2627 · ESIC
University

Antonio Tello  
septiembre 2026

# Cómo usar este notebook

Este notebook **no viene resuelto**. Mientras en clase se proyecta la
presentación, tú lo vas completando en tu repo
`grdan-series-temporales`.

- **`___`** es un hueco: sustitúyelo por el código que falta. Si
  ejecutas una celda con un hueco sin rellenar, dará error; es lo
  esperado.
- **✍️ Tu respuesta:** celda de texto para contestar con tus palabras.
  Doble clic para editarla.
- Cada paso lleva un código (**A1**, **B2**, **C3**…) que coincide con
  la diapositiva donde está resuelto.

Pasos previos:

1.  Copia este notebook en la carpeta `notebooks/` de tu repo y ábrelo
    en VS Code.
2.  **Select Kernel → `.venv`**.
3.  Ejecuta las celdas **en orden** con `Shift+Enter`.

| Bloque | Qué                                                         | Hoy |
|--------|-------------------------------------------------------------|-----|
| A      | Repaso de pandas: `DataFrame`, fechas, selección, `groupby` | 🟢  |
| B      | Varias series en una tabla (`unique_id`, `ds`, `y`)         | 🟢  |
| C      | El gráfico temporal y los patrones                          | 🟢  |
| D      | Gráficos estacionales y de subseries                        | 🟡  |
| E      | Cambiar la frecuencia: `resample` y `rolling`               | 🟡  |
| F      | Retardos, autocorrelación (ACF) y ruido blanco              | 🟡  |

## S1 · Imports y configuración

Esta celda viene completa. Ejecútala sin más.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "figure.dpi": 100,
    "figure.constrained_layout.use": True,
})
sns.set_style("whitegrid")
pd.set_option("display.precision", 2)

## S2 · Una función para cargar datos

También viene completa: la usaremos todo el curso. Descarga el CSV del
libro la primera vez, lo guarda en `data/` y las siguientes veces lo lee
del disco.

In [2]:
URL_DATOS = "https://otexts.com/fpppy/data/"

# data/ va en la raíz del repo (donde está pyproject.toml)
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists())
DATA = raiz / "data"

def cargar(nombre, **kwargs):
    """Lee data/<nombre>; si no existe, lo descarga de FPPPy y lo guarda."""
    ruta = DATA / nombre
    if not ruta.exists():
        DATA.mkdir(exist_ok=True)
        # la web del libro rechaza peticiones sin User-Agent (error 403)
        pd.read_csv(URL_DATOS + nombre,
                    storage_options={"User-Agent": "Mozilla/5.0"}
                    ).to_csv(ruta, index=False)
        print(f"Descargado {nombre} → {ruta}")
    return pd.read_csv(ruta, **kwargs)

## S3 · Que los datos no acaben en GitHub

Abre `.gitignore` y añade una línea con `data/`. Después, en la
terminal, ejecuta `git status`.

✍️ **Tu respuesta:** ¿por qué no deben subirse los CSV al repositorio?
No hay que subir los CSV al repositorio de github, porque ocupan mucho espacio y los datos se pueden descargar desde la web la primera vez que ejecutas el codigo, y tambien, por eso hemos metido data en el gitignore, para que los datos se queden en nuestro ordenador.

# A · Repaso de pandas 🟢

## A1 · El `DataFrame`: índice, columnas y tipos

In [3]:
df = pd.DataFrame({
    "anio": [2015, 2016, 2017, 2018, 2019],
    "valor": [123, 39, 78, 52, 110],
})
df

,anio,valor
0,2015,123
1,2016,39
2,2017,78
3,2018,52
4,2019,110


Muestra el índice, las columnas y el tipo de cada columna:

In [8]:
print(df.index)
print(df.columns)
df.dtypes

RangeIndex(start=0, stop=5, step=1)
Index(['anio', 'valor'], dtype='str')


anio     int64
valor    int64
dtype: object

Convierte la columna `anio` en el índice:

In [10]:
df.set_index("anio")

,valor
anio,
2015,123
2016,39
2017,78
2018,52
2019,110


✍️ **Tu respuesta:** ¿qué ha cambiado en la tabla al usar `anio` como
índice?
lo que pasa es que año pasa de ser una columna a ser el indice de las filas
## A2 · Fechas: `Timestamp` y `Period`

In [11]:
print(repr(pd.Timestamp("2020-01")))
print(repr(pd.Period("2020-01")))
print(repr(pd.Period("2020-01-15", freq="Q")))

Timestamp('2020-01-01 00:00:00')
Period('2020-01', 'M')
Period('2020Q1', 'Q-DEC')


✍️ **Tu respuesta:** ¿qué diferencia hay entre un `Timestamp` y un
`Period`? ¿Qué representa el último?
Timesr¡tamp, representa un momento puntual en el tiempo. Sin embargo, period representa un periodo de tiempo y la letra que sale al lado dice si el periodo es de un año, semana, etc. 

## A3 · Secuencias de fechas y el accesor `.dt`

Crea 6 fechas separadas **8 horas** entre sí, empezando el 1 de enero de
2020:

In [13]:
rejilla = pd.date_range("2020-01-01", periods=6, freq="8h")
rejilla

DatetimeIndex(['2020-01-01 00:00:00', '2020-01-01 08:00:00',
               '2020-01-01 16:00:00', '2020-01-02 00:00:00',
               '2020-01-02 08:00:00', '2020-01-02 16:00:00'],
              dtype='datetime64[us]', freq='8h')

Extrae de cada fecha el año, el trimestre, el nombre del día y el mes
como periodo:

In [15]:
d = pd.DataFrame({"ds": pd.date_range("2024-12-30", periods=5, freq="D")})
d.assign(
    anio=d["ds"].dt.year,
    trimestre=d["ds"].dt.quarter,
    dia_semana=d["ds"].dt.day_name(),
    mes=d["ds"].dt.to_period("M"),
)

,ds,anio,trimestre,dia_semana,mes
0,2024-12-30,2024,4,Monday,2024-12
1,2024-12-31,2024,4,Tuesday,2024-12
2,2025-01-01,2025,1,Wednesday,2025-01
3,2025-01-02,2025,1,Thursday,2025-01
4,2025-01-03,2025,1,Friday,2025-01


## A4 · Leer un CSV con fechas: el caso PBS

Recetas de medicamentos subvencionados en Australia (el **Caso 2** del
encuadre). La columna de fechas se llama `Month`.

In [17]:
pbs = cargar("PBS_unparsed.csv", parse_dates=["Month"])
print(pbs.shape)
pbs.dtypes

Descargado PBS_unparsed.csv → c:\Users\Eva\dev\grdan-series-temporales\data\PBS_unparsed.csv
(67596, 9)


Month         datetime64[us]
Concession               str
Type                     str
ATC1                     str
ATC1_desc                str
ATC2                     str
ATC2_desc                str
Scripts                int64
Cost                 float64
dtype: object

Ahora léelo **sin** `parse_dates` y mira los tipos:

In [18]:
pbs_sin = cargar("PBS_unparsed.csv")
pbs_sin.dtypes

Month             str
Concession        str
Type              str
ATC1              str
ATC1_desc         str
ATC2              str
ATC2_desc         str
Scripts         int64
Cost          float64
dtype: object

✍️ **Tu respuesta:** ¿qué tipo tiene `Month` en cada caso? ¿Qué
problemas aparecerían más adelante con la segunda versión?
en el primero es datetyme y el segundo str, el problema que habria es que si tu pones el numero del mes, habria error. 

## A5 · Filtrar, agrupar y agregar

Objetivo: el coste total mensual de los antidiabéticos
(`ATC2 == "A10"`), en millones de A\$.

In [20]:
a10 = (
    pbs.loc[pbs["ATC2"] == "A10"]
    .groupby("Month", as_index=False)
    .agg(coste=("Cost", "sum"))
    .assign(coste=lambda x: x["coste"] / 1e6)
)
a10.head()

,Month,coste
0,1991-07-01,3.53
1,1991-08-01,3.18
2,1991-09-01,3.25
3,1991-10-01,3.61
4,1991-11-01,3.57


## A6 · Seleccionar por fechas

Con la fecha como índice se puede seleccionar escribiendo la fecha como
texto. Selecciona todo 2005 y, después, de junio a diciembre de 2007:

In [ ]:
a10_idx = a10.set_index("Month")
a10_idx.loc[___]

In [ ]:
a10_idx.loc[___:___]

## EjA · Ejercicio A (5’)

Calcula el **número total de recetas** (`Scripts`) por mes del grupo
`ATC2 == "H02"` (corticoides). ¿En qué mes hubo más recetas? Pista:
`.idxmax()`.

In [ ]:
# tu código aquí

✍️ **Tu respuesta:** mes con más recetas y cuántas fueron.

# B · Varias series en una tabla 🟢

## B1 · Variables clave: ¿cuántas series hay?

Pasajeros semanales de Ansett, la aerolínea del **Caso 4** del encuadre.

In [ ]:
ansett = cargar("ansett.csv", parse_dates=["ds"])
ansett.head()

Cada combinación única de `Airports` y `Class` es una serie distinta:

In [ ]:
ansett[["Airports", "Class"]].___()

✍️ **Tu respuesta:** ¿cuántas series hay en la tabla? ¿Por qué no son 10
× 3?

## B2 · El formato de Nixtla: `unique_id`, `ds`, `y`

| Columna     | Qué es                |
|-------------|-----------------------|
| `unique_id` | qué serie es          |
| `ds`        | cuándo (*date stamp*) |
| `y`         | el valor observado    |

Crea `unique_id` pegando aeropuerto y clase con `" · "` y cuenta cuántos
hay:

In [ ]:
ansett = ansett.assign(
    unique_id=ansett["Airports"] + " · " + ___
)
ansett["unique_id"].___()

## B3 · Largo ↔ ancho: `pivot` y `melt`

Pon las tres clases de la ruta `MEL-SYD` en columnas (formato ancho):

In [ ]:
ancho = (
    ansett.loc[ansett["Airports"] == "MEL-SYD"]
    .pivot(index=___, columns=___, values=___)
)
ancho.head()

Y de vuelta a formato largo:

In [ ]:
(ancho.reset_index()
      .melt(id_vars="ds", var_name="Class", value_name="y")
      .head())

✍️ **Tu respuesta:** ¿cuándo te conviene el formato ancho y cuándo el
largo?

# C · El gráfico temporal 🟢

## C1 · Lo primero, siempre: dibujar la serie

Dibuja la clase económica de Melbourne–Sídney, en miles de pasajeros:

In [ ]:
mel_syd = (
    ansett.loc[(ansett["Airports"] == ___)
               & (ansett["Class"] == ___)]
    .assign(y=lambda x: x["y"] / 1000)
)

fig, ax = plt.subplots()
ax.plot(mel_syd[___], mel_syd[___], color="black")
ax.set(title="Ansett · económica MEL–SYD",
       xlabel="Semana", ylabel="Pasajeros (miles)");

✍️ **Tu respuesta:** enumera **al menos cuatro** rasgos de esta serie
que un modelo de predicción tendría que tener en cuenta.

## C2 · La forma “Nixtla” de dibujar

`plot_series` dibuja directamente una tabla en formato largo, una serie
por panel. Esta celda viene completa:

In [ ]:
from utilsforecast.plotting import plot_series

mel = ansett.loc[ansett["Airports"] == "MEL-SYD"]
plot_series(mel[["unique_id", "ds", "y"]])

## C3 · Tendencia, estacionalidad y ciclo

| Patrón | Definición |
|------------------------------------|------------------------------------|
| **Tendencia** | Subida o bajada a largo plazo; no tiene por qué ser lineal |
| **Estacionalidad** | Se repite con un **periodo fijo y conocido**, ligado al calendario |
| **Ciclo** | Subidas y bajadas **sin periodo fijo**, normalmente de 2 años o más |

Dibuja el coste mensual de antidiabéticos (`a10`, del paso A5):

In [ ]:
fig, ax = plt.subplots()
ax.plot(___, ___, color="black")
ax.set(title="Antidiabéticos en Australia",
       xlabel="Mes", ylabel="Millones de A$");

✍️ **Tu respuesta:** completa la tabla.

| Patrón         | ¿Aparece? | ¿Cómo es? |
|----------------|-----------|-----------|
| Tendencia      |           |           |
| Estacionalidad |           |           |
| Ciclo          |           |           |

¿Qué crees que causa la caída de cada enero?

## EjC · Ejercicio C (5’)

Producción trimestral en Australia. Dibuja `Beer`, `Bricks` y
`Electricity`, un gráfico para cada una.

In [ ]:
prod = cargar("aus_production.csv", parse_dates=["ds"])
prod.head()

In [ ]:
# tu código aquí

✍️ **Tu respuesta:**

| Serie       | Tendencia | Estacionalidad | Ciclo | Algo raro |
|-------------|-----------|----------------|-------|-----------|
| Beer        |           |                |       |           |
| Bricks      |           |                |       |           |
| Electricity |           |                |       |           |

------------------------------------------------------------------------

> 🟢 **Hasta aquí, el mínimo de hoy.** Guarda y sube tu trabajo:
> `git add notebooks/ .gitignore` ·
> `git commit -m "nb02: pandas y time plots"` · `git push`

------------------------------------------------------------------------

# D · Gráficos estacionales 🟡

## D1 · Seasonal plot: un año encima de otro

En el eje X, el mes; una línea por año:

In [ ]:
d = a10.assign(anio=a10["Month"].dt.year,
               mes=a10["Month"].dt.month)

fig, ax = plt.subplots()
sns.lineplot(data=d, x=___, y="coste", hue=___,
             palette="viridis", legend=False, ax=ax)
ax.set(title="Seasonal plot · antidiabéticos",
       xlabel="Mes", ylabel="Millones de A$",
       xticks=range(1, 13));

✍️ **Tu respuesta:** ¿qué mes rompe el patrón todos los años? ¿Ves algo
raro en 2008?

## D2 · Subseries plot: cada mes, su mini-serie

Un panel por mes; dentro, la evolución de ese mes a lo largo de los años
y una línea con su media:

In [ ]:
fig, axes = plt.subplots(1, 12, figsize=(11, 3), sharey=True)
for ax, (mes, sub) in zip(axes, d.groupby(___)):
    ax.plot(sub["anio"], sub["coste"], color="black", lw=1)
    ax.axhline(sub["coste"].___(), color="tab:blue", lw=1)
    ax.set_title(mes)
    ax.set_xticks([])
fig.suptitle("Subseries plot · antidiabéticos");

✍️ **Tu respuesta:** ¿qué ves aquí que no se veía en el seasonal plot?

## D3 · Estacionalidades que no coinciden

Viajes de vacaciones por estado australiano, por trimestre:

In [ ]:
tourism = cargar("tourism.csv", parse_dates=["ds"])
vacaciones = (
    tourism.loc[tourism["Purpose"] == ___]
    .groupby(["State", "ds"], as_index=False)["y"].sum()
    .assign(trimestre=lambda x: "Q" + x["ds"].dt.quarter.astype(str),
            anio=lambda x: x["ds"].dt.year)
)

g = sns.relplot(data=vacaciones, x="trimestre", y="y", hue="anio",
                col=___, col_wrap=4, kind="line", palette="viridis",
                height=2.2, facet_kws={"sharey": False}, legend=False)
g.set_titles("{col_name}");

✍️ **Tu respuesta:** ¿en qué trimestre tiene su pico cada estado? ¿Por
qué no es el mismo? ¿Qué pasaría si sumaras todos los estados?

# E · Cambiar la frecuencia 🟡

## E1 · `resample`: de medias horas a días y meses

Demanda eléctrica de Victoria cada 30 minutos (2012–2014):

In [ ]:
elec = (
    cargar("vic_elec.csv", parse_dates=["ds"])
    .query("unique_id == 'Demand'")
    .set_index("ds")["y"]
)
print(len(elec), elec.index.min(), elec.index.max())

Dibuja la misma serie a tres frecuencias. Para la diaria usa `"D"` y
para la mensual `"MS"` (*month start*), sumando:

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 7))
elec.loc["2014-01-06":"2014-01-19"].plot(ax=axes[0], title="Cada 30 min")
elec.resample(___).___().plot(ax=axes[1], title="Diaria")
elec.resample(___).___().plot(ax=axes[2], title="Mensual")
for ax in axes:
    ax.set_xlabel("")

✍️ **Tu respuesta:** ¿qué patrón estacional se ve en cada una de las
tres frecuencias?

## E2 · `rolling`: medias móviles

In [ ]:
diaria = elec.resample("D").sum() / 1000   # GWh

fig, ax = plt.subplots()
diaria.plot(ax=ax, color="lightgray", label="diaria")
diaria.rolling(___).mean().plot(ax=ax, label="media móvil 7 días")
diaria.rolling(___, center=True).mean().plot(
    ax=ax, label="media móvil 30 días (centrada)")
ax.set(ylabel="GWh", xlabel="")
ax.legend();

✍️ **Tu respuesta:** `center=True` usa, para cada día, los 15 días
anteriores y los 15 posteriores. ¿Qué problema tendría usar esa columna
como variable de entrada de un modelo que predice el día siguiente?

# F · Retardos y autocorrelación 🟡

## F1 · Lag plots: la serie contra sí misma

Producción de cerveza desde 2000. Cada panel enfrenta $y_t$ con
$y_{t-k}$:

In [ ]:
beer = (
    prod.loc[prod["ds"] >= "2000", ["ds", "Beer"]]
    .rename(columns={"Beer": "y"})
    .assign(Q=lambda x: "Q" + x["ds"].dt.quarter.astype(str))
    .reset_index(drop=True)
)

fig, axes = plt.subplots(3, 3, figsize=(8, 7), sharex=True, sharey=True)
for k, ax in enumerate(axes.flat, start=1):
    sns.scatterplot(x=beer["y"].shift(___), y=beer["y"], hue=beer["Q"],
                    ax=ax, legend=(k == 3), palette="viridis")
    ax.set(title=f"retardo {k}", xlabel="", ylabel="")

✍️ **Tu respuesta:** ¿en qué retardos la nube se alinea sobre la
diagonal? ¿En cuáles la relación es negativa? ¿Por qué?

## F2 · La ACF: todos los lag plots en un gráfico

La autocorrelación en el retardo $k$ mide la relación lineal entre $y_t$
e $y_{t-k}$:

$$r_k = \frac{\sum_{t=k+1}^{T}(y_t-\bar y)(y_{t-k}-\bar y)}{\sum_{t=1}^{T}(y_t-\bar y)^2}$$

Dibuja la ACF de la cerveza hasta el retardo 16:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
plot_acf(beer[___], lags=___, zero=False,
         bartlett_confint=False, auto_ylims=True, ax=ax)
ax.set(title="ACF · cerveza", xlabel="retardo (trimestres)");

✍️ **Tu respuesta:** ¿qué barras destacan y cómo encajan con los lag
plots de F1?

## F3 · Tendencia y estacionalidad en la ACF

ACF del coste de antidiabéticos hasta el retardo 48:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
plot_acf(a10[___], lags=___, zero=False,
         bartlett_confint=False, auto_ylims=True, ax=ax)
ax.set(title="ACF · antidiabéticos", xlabel="retardo (meses)");

✍️ **Tu respuesta:** describe la forma de la ACF. ¿Qué parte se debe a
la tendencia y cuál a la estacionalidad?

## F4 · Ruido blanco

Una serie **sin autocorrelación** es ruido blanco. Generamos 50 valores
normales independientes:

In [ ]:
rng = np.random.default_rng(30)
ruido = pd.Series(rng.normal(0, 1, ___))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
ruido.plot(ax=axes[0], color="black", title="Ruido blanco")
plot_acf(ruido, lags=16, zero=False, bartlett_confint=False,
         auto_ylims=True, ax=axes[1], title="Su ACF");

En ruido blanco se espera que el 95 % de las barras quede dentro de
$\pm 1{,}96/\sqrt{T}$.

✍️ **Tu respuesta:** calcula ese límite para $T = 50$. ¿Cuántas barras
lo superan? ¿Qué predicción harías para una serie que es ruido blanco?

## EjF · Ejercicio F (para casa)

Del libro, §2.10. Dibuja la serie **y** la ACF de:

1.  `Bricks` (producción trimestral de ladrillos, `prod`). Ojo: tiene
    valores perdidos al final; usa `.dropna()`.
2.  La demanda eléctrica **diaria** de Victoria (`diaria`, del paso E2).

In [ ]:
# tu código aquí

✍️ **Tu respuesta:** para cada serie, ¿tendencia? ¿Estacionalidad? ¿De
qué periodo? ¿Es ruido blanco?

# Para cerrar

1.  Guarda el notebook **con las salidas**.
2.  `git add notebooks/ .gitignore` · `git commit -m "nb02"` ·
    `git push`.
3.  Comprueba en GitHub que `data/` **no** aparece en el repo.